# NLP Practical 7 — Natural Language Generation

Run in **Google Colab**. Cells with `pip install` only need to run once per session.

**Covers:** a full content-determination -> surface-realization pipeline on synthetic live-style weather data, template-based NLG, statistical n-gram text generation, and a short survey of real-world NLG deployments.


In [ ]:
!pip install nltk -q
import nltk
nltk.download("punkt")
import random


In [ ]:
# ============================================================
# PART A: Content Determination -> Realization pipeline on (synthetic) weather data
# ============================================================
# Synthetic "live" weather record, standing in for a weather-API feed
weather_data = {
    "city": "Bathinda", "condition": "partly cloudy", "temp_c": 34,
    "humidity": 48, "wind_kmh": 12, "forecast_tomorrow": "clear skies"
}

def content_determination(data):
    """Decide WHAT to say: pick the salient facts."""
    facts = []
    facts.append(("condition", data["city"], data["condition"]))
    facts.append(("temperature", data["city"], data["temp_c"]))
    if data["wind_kmh"] > 10:
        facts.append(("wind", data["city"], data["wind_kmh"]))
    facts.append(("forecast", data["city"], data["forecast_tomorrow"]))
    return facts

def surface_realization(facts):
    """Decide HOW to say it: map facts to fluent sentences."""
    sentence_templates = {
        "condition": "{0} is currently {2}.",
        "temperature": "The temperature is {2}\u00b0C.",
        "wind": "Winds are blowing at {2} km/h.",
        "forecast": "Tomorrow's forecast: {2}.",
    }
    return " ".join(sentence_templates[f[0]].format(*f) for f in facts)

facts = content_determination(weather_data)
print("Content-determination output (facts):")
for f in facts: print(" ", f)

print("\nRealized text:")
print(surface_realization(facts))


In [ ]:
# ============================================================
# PART B: Template-based NLG
# ============================================================
templates = [
    "{city} sees {condition} today, with highs near {temp_c} degrees.",
    "Expect {condition} in {city}; temperatures around {temp_c}\u00b0C.",
]

for t in templates:
    print(t.format(**weather_data))


In [ ]:
# ============================================================
# PART C: Statistical n-gram text generation
# ============================================================
import random, nltk
from collections import defaultdict

corpus = """the weather today is warm and sunny the sky is clear and the wind is calm
the weather tomorrow will be cloudy and cool the wind will be strong
the forecast shows clear skies and warm temperatures across the region"""

tokens = nltk.word_tokenize(corpus.lower())
bigrams = list(nltk.bigrams(tokens))

model = defaultdict(list)
for w1, w2 in bigrams:
    model[w1].append(w2)

def generate(seed, n_words=12):
    word = seed
    out = [word]
    for _ in range(n_words - 1):
        choices = model.get(word)
        if not choices:
            break
        word = random.choice(choices)
        out.append(word)
    return " ".join(out)

random.seed(7)
print("Bigram-generated text:")
print(generate("the"))


### Real-world NLG survey (discussion notes)

- **Automated journalism** — Associated Press has used Automated Insights' Wordsmith (and
  similar Arria-style engines) to generate thousands of corporate earnings reports and
  Minor League Baseball recaps per season, freeing reporters for investigative work.
- **BI dashboards** — Tools like Narrative Science / Quill turn a chart's underlying numbers
  into a plain-English paragraph automatically.
- **Modern shift** — Large language models now often perform content-determination and
  surface-realization *jointly* in one forward pass, collapsing the classic NLG pipeline shown
  above into a single learned model. Understanding the pipeline stages still matters for
  debugging *why* a generated report emphasizes the wrong fact, or hallucinates one.
